# Somo la 18: Kuweka Usalama wa Maajenti wa AI kwa Kupokea Kificho cha Usimbaji

## Daftari la Vitendo

Daftari hili linaelezea mazoezi manne:

1. **Saini risiti yako ya kwanza** kwa simu ya chombo cha ajenti na uithibitishe.
2. **Badilisha risiti** na angalia uthibitisho ukishindikana.
3. **Jenga mnyororo wa risiti tatu** na thibitisha usahihi wa mnyororo.
4. **Funga simu ya chombo cha Mfumo wa Maajenti wa Microsoft** kiasi kila hatua itokeza risiti.

Misingi yote ya usimbaji inazalishwa kutoka maktaba zinazotunzwa vyema (`pynacl` kwa Ed25519, `jcs` kwa RFC 8785 canonical JSON, `hashlib` kutoka maktaba ya kawaida ya Python kwa SHA-256). Mantiki ya risiti yenyewe ni Python safi ambayo unaweza kusoma na kurekebisha.

Endesha seli kwa mpangilio. Kila sehemu ni fupi na yenye kujitegemea.


## Setup

Sakinisha utegemezi wawili. Vyote vina leseni zinazoruhusu matumizi (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Vifaa Msaidizi

Vifaa hivi viwili hufanya usimbaji wa base64url (bila kujaza) na ufupishaji wa SHA-256 wa vitu vyovyote. Huvifanya sehemu nyingine ya daftari kuuendeleze kuzingatia mantiki ya risiti yenyewe.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sehemu 1: Saini risiti yako ya kwanza

Fikiria wakala wetu wa **Contoso Travel** amechukua ndege kutoka Sydney kwenda Los Angeles kwa mteja. Tunataka kurekodi wito huu wa kifaa kama risiti iliyosainiwa ili mkaguzi wa baadaye aweze kuithibitisha bila kutuamini sisi.

### Hatua 1.1: Tengeneza ufunguo wa kusaini

Katika uzalishaji, ufunguo wa kusaini wa wakala ungeishi katika kifaa cha usalama wa vifaa (HSM), Azure Key Vault, au duka lililolindwa kama hilo. Kwa somo hili tunatengeneza ufunguo mpya kwenye kumbukumbu.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Hatua 1.2: Jenga mzigo wa risiti

Mzigo unajumuisha kila kitu tunachotaka risiti ithibitishe: nani alitenda, zana gani, kwa hoja gani, nini kilirudiwa, chini ya sera gani, na lini. Tunafanya hash ya hoja na matokeo badala ya kuzijumlisha moja kwa moja ili risiti isitumie maudhui nyeti.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Hatua 1.3: Saini na kusanyisha risiti

Hatua tatu:

1. Fanya payload kuwa ya kawaida ukitumia JCS ili utekelezaji wawili wanaoleta risiti sawa kiakili wazalishe baiti sawa kabisa.
2. Fanya hash ya baiti za kawaida kwa kutumia SHA-256.
3. Saini hash hiyo kwa kutumia ufunguo binafsi wa Ed25519.

Sahihi kisha inaambatanishwa kwenye payload ya asili kutengeneza risiti ya mwisho.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Hatua 1.4: Thibitisha risiti

Uthibitishaji unageuza mchakato. Tunatoa saini, kuhesabu upya hash halali, na kuangalia saini dhidi ya ufunguo wa umma kwenye risiti.

Mkaguzi anayefanya uthibitishaji huu hahitaji chochote kutoka kwetu isipokuwa risiti yenyewe. Hakuna huduma ya kupigia simu, hakuna saraka ya funguo ya kuulizia, hakuna imani inayohitajika.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Unapaswa kuona `Receipt is valid: True`. Wakala amezaa rekodi yake ya kwanza ya ukaguzi iliyotiwa saini kwa kificho.


## Sehemu ya 2: Kusababisha mabadiliko kwenye risiti

Madhumuni yote ya risiti ni kwamba zinaonyesha wazi unapojaribu kuzibadilisha. Hebu tuonyeshe hivyo.

Tutabadilisha herufi moja tu kwenye risiti na tutaangalia uthibitishaji ukishindikana.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Nini kilitokea sasa hivi?

Tulipobadilisha `policy_id`, bayt za kawaida zilibadilika. Hashi ya SHA-256 ya bayt hizo ilibadilika. Saini (ambayo ilikuwa juu ya hashi ya awali) haiko tena sawa na hashi mpya. Uthibitishaji unarudisha `False` kwa usahihi.

Hakuna njia ya kubadilisha kipengele chochote cha risiti na bado kuithibitisha, isipokuwa mshambuliaji ana ufunguo binafsi. Mradi tu ufunguo binafsi uko kwenye hifadhi ya funguo na ufunguo wa umma umechapishwa, kuiba hakutawezekana kufichwa.

Jaribu mwenyewe: badilisha `tool_name` au `agent_id` au `timestamp` katika seli ya juu na endesha tena. Kila mabadiliko husababisha risiti isiyo halali.


## Sehemu ya 3: Kuhuisha risiti pamoja

Risiti moja hulinda kitendo kimoja. Wakala wengi huchukua vitendo vingi. Ili kufanya mchakato mzima kuonekana umeharibika, tunahusisha kila risiti na ile ya awali kwa kujumuisha hash ya risiti ya awali kwenye maelezo ya risiti mpya.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ikiwa mtu yeyote ataondoa au kuandaa upya risiti, mnyororo unavunjika sehemu hiyo tu. Uthibitishaji wa risiti yoyote baadaye huwakwama kwa sababu `previous_receipt_hash` haiendani tena na hash halisi ya risiti iliyo mbele yake.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Sasa vunja pete kwa kubadilisha risiti ya katikati na thibitisha upya. Risiti iliyodanganywa inakumbwa na kushindwa kwenye ukaguzi wa saini yake, NA risiti inayofuata inashindwa kwenye ukaguzi wa kiungo cha mnyororo (kwa sababu `previous_receipt_hash` yake haizilingani tena na hash ya risiti ya katikati iliyobadilishwa).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Risiti 0 bado inathibitishwa (hakuathiriwa na haina risiti ya awali ya kutegemea). Risiti 1 inashindwa kwenye ukaguzi wa saini yake kwa sababu tulibadilisha `tool_args_hash`. Risiti 2 inashindwa kwenye ukaguzi wa kiungo cha mnyororo kwa sababu `previous_receipt_hash` yake ilihesabiwa dhidi ya risiti ya awali 1 (iliyobadilishwa sasa).

Hata kama mshambuliaji atasaini tena risiti iliyobadilishwa 1 (ambayo hawawezi kufanya bila ufunguo binafsi), kutokubaliana kwa kiungo cha mnyororo katika risiti 2 bado kungefunua uharibu huo. Kuficha mabadiliko, mshambuliaji angenahitaji kusaini tena kila risiti kuanzia mahali pa mabadiliko na kuendelea, jambo ambalo linahitaji umiliki wa ufunguo binafsi.


## Sehemu 4: Tamatisha wito wa zana ya wakala kwa kusaini risiti

Katika utekelezaji halisi, hutaki kila mwandishi wa wakala kukumbuka kuita `make_receipt`. Unataka usaini wa risiti kuwa wa moja kwa moja kwa kila mwito wa zana.

Hapa kuna muundo rahisi kabisa: darasa la kuzuia ambalo linachukua kazi yoyote ya zana inayoweza kuitwa na kurudisha toleo linalotolewa risiti. Hii inaendana na mfumo wowote wa wakala, ikiwa ni pamoja na Microsoft Agent Framework (`agent_framework.azure`).

Kama huna mradi wa Azure AI Foundry uliowekwa, mfano wa ndani hapa chini bado unaonyesha muundo huo.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Kuunganisha na Microsoft Agent Framework

Kifuniko cha `ReceiptedTool` kilicho hapo juu hakitegemei framework yoyote. Ili kukitumia ndani ya wakala aliyojengwa kwa Microsoft Agent Framework, sajili kazi iliyofunikwa kama chombo. Mchoro (utabadilisha mock na usajili halisi wa chombo cha Azure AI Foundry):

```python
# Pseudocode inaonyesha umbo la uvumilivu.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Wewe ni wakala wa Usafiri wa Contoso ...",
#     tools=[receipted_lookup],   # chombo kilichofunikwa, si kazi mbichi
# )
# response = agent.run("Tafuta ndege kutoka Sydney kwenda Los Angeles mwezi Juni.")
#
# # Baada ya utekelezaji, kila mwito wa chombo aliofanya wakala una risiti iliyosainiwa:
# audit_chain = receipted_lookup.receipts
```

Framework ya wakala haihitaji kujua chochote kuhusu risiti. Kusaini risiti kunafunikwa kuzunguka chombo, haliwezi kuingizwa moja kwa moja kwenye framework. Hivi ndivyo unavyoongeza asili kwenye msimbo wa wakala uliopo bila kuandika upya wakala huo.


## Muhtasari na changamoto ya kupanua

Umekamilisha:

- Kutengeneza jozi ya funguo za Ed25519.
- Kujenga na kusaini risiti kwa wito wa zana ya wakala.
- Kuhakiki risiti kimtandaoni kwa kutumia tu ufunguo wa umma.
- Kubadilisha risiti na kuona uthibitisho ukishindikana.
- Kujenga mfululizo ulio na minyororo ya hash ya risiti tatu.
- Kubadilisha sehemu ya katikati ya mnyororo na kuona kushindwa kwa saini na pia kushindwa kwa kiungo cha mnyororo.
- Kufunga kazi ya zana na kusaini risiti moja kwa moja.

**Changamoto ya kupanua.** Panua muundo wa risiti kwa uwanja wa `request_id` (UUID kwa ajili ya ufuatiliaji wa usambazaji). Sasisha `make_receipt` kuijumuisha, na hakikisha risiti bado zinathibitishwa kufikia mwisho. Kisha badilisha uwanja huo baada ya kusaini na hakikisha uthibitisho unashindwa. Hii inakugeuza kuelewa jinsi kila byte ya usimbaji wa kawaida unavyosaidia kwenye saini.

**Mwangwi muhimu.** Risiti zinathibitisha mambo matatu tu na matatu tu: utambulisho (funguo hii ilisaini maudhui haya), uadilifu (maudhui hayajabadilika tangu kusaini), na mpangilio (risiti hii ilikuja baada ya risiti ile). Hazithibitishi kuwa kitendo cha wakala kilikuwa sahihi, kuwa sera iliyopewa jina katika `policy_id` ilitathminiwa kweli, au wakala alifuata kanuni zote. Risiti ni msingi. Usimamizi ni mfumo unaoujenga juu yake.

Soma README ya somo tena ukiwa na mwangwi huo akili. Makosa ya kawaida ya timu kuhusu risiti ni kudhani "tuna risiti" maana yake ni "tunadhibitiwa." Hilo si sahihi. Risiti hufanya tabia ya wakala iweze kukaguliwa. Haziifanyi kuwa sahihi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
